# 📏 Sarcasm Detection — Method 1: Rule-Based (spaCy)
**Project:** `Sarcasm-Detection-NLP` | **Course:** CSE3246 — Natural Language Processing  
**Dataset:** `raquiba/Sarcasm_News_Headline` (~28K headlines, HuggingFace)  
**Notebook:** `01_rule_based.ipynb` — Rule Engine → 10 Rubric Metrics → Export JSON

---
### Notebook Architecture
```
Section 1 │ Environment & Imports
Section 2 │ Data Loading  (identical pipeline to 02_tfidf_sklearn.ipynb)
Section 3 │ Rule Engine — 7 linguistic signals (spaCy)
Section 4 │ Prediction on test split
Section 5 │ Full Evaluation — 10 Rubric Metrics  (same compute_all_metrics as M2/M3)
Section 6 │ Visualisations
Section 7 │ Error Analysis
Section 8 │ Export JSON  (compatible with 03_bert_finetune.ipynb cross-comparison)
```

## Section 1 — Environment Setup & Imports

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────────
!pip install spacy datasets matplotlib seaborn scikit-learn pandas numpy --quiet
!python -m spacy download en_core_web_sm -q

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import re, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import spacy
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, matthews_corrcoef
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
RANDOM_STATE = 42   # ← same seed used in 02_tfidf_sklearn.ipynb and 03_bert_finetune.ipynb
np.random.seed(RANDOM_STATE)

# ── Output directories — mirrors 02_tfidf_sklearn.ipynb layout ───────────────
OUTPUT_DIR  = Path('outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
M1_DIR      = OUTPUT_DIR / 'method1'
for d in [OUTPUT_DIR, FIGURES_DIR, M1_DIR]:
    d.mkdir(parents=True, exist_ok=True)

nlp = spacy.load('en_core_web_sm', disable=['parser'])  # NER + tagger only

print('✅ All imports successful.')
print(f'   spaCy model : {nlp.meta["name"]}')
print(f'📁 Outputs → {OUTPUT_DIR.resolve()}')

## Section 2 — Data Loading

> **Identical pipeline to `02_tfidf_sklearn.ipynb`** — same HuggingFace load, same column detection,
> same `df` variable name, same `RANDOM_STATE=42` stratified 80/20 split.
> This ensures the test set is identical across all three methods for a fair comparison.

In [ ]:
# ── Load from HuggingFace Hub ─────────────────────────────────────────────────
print('⏳ Fetching raquiba/Sarcasm_News_Headline from HuggingFace Hub...')
t0 = time.time()
raw = load_dataset('raquiba/Sarcasm_News_Headline')
print(f'   Done in {time.time()-t0:.1f}s')

# Combine all splits into a single DataFrame
dfs = []
for split_name, split_data in raw.items():
    tmp = split_data.to_pandas()
    tmp['split'] = split_name
    dfs.append(tmp)
df_raw = pd.concat(dfs, ignore_index=True)

print(f'\n📊 Raw dataset shape: {df_raw.shape}')
print(f'   Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# ── Schema inspection + clean working DataFrame ───────────────────────────────
# Same column-detection logic as 02_tfidf_sklearn.ipynb
label_col = 'is_sarcastic' if 'is_sarcastic' in df_raw.columns else df_raw.columns[-1]
text_col  = 'headline'     if 'headline'     in df_raw.columns else df_raw.columns[0]
print(f'→ Text column  : "{text_col}"')
print(f'→ Label column : "{label_col}"')

df = df_raw[[text_col, label_col]].copy()
df.columns = ['headline', 'label']
df = df.dropna(subset=['headline', 'label'])
df['label'] = df['label'].astype(int)
df = df.reset_index(drop=True)

print(f'\nWorking dataset : {len(df):,} rows')
print(f'Sarcastic (1)   : {df["label"].sum():,} ({df["label"].mean()*100:.1f}%)')
print(f'Non-sarcastic(0): {(df["label"]==0).sum():,} ({(df["label"]==0).mean()*100:.1f}%)')
df.head(3)

In [ ]:
# ── Stratified 80/20 split — IDENTICAL to 02_tfidf_sklearn.ipynb ─────────────
# Rule engine needs no training set, but we keep the same split so test sets match.
X = df['headline'].values   # raw text — rule engine uses this directly
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train : {len(X_train):,} samples  (unused by rule engine; kept for split consistency)')
print(f'Test  : {len(X_test):,}  samples  (sarcastic: {y_test.sum():,} | {y_test.mean()*100:.1f}%)')

## Section 3 — Rule Engine (7 Linguistic Signals)

In [ ]:
# ── Lexicons & Regex Patterns ─────────────────────────────────────────────────

HYPERBOLE_WORDS = {
    'literally', 'absolutely', 'totally', 'completely', 'utterly', 'definitely',
    'obviously', 'clearly', 'certainly', 'undeniably', 'unbelievably', 'insanely',
    'hilariously', 'miraculously', 'shockingly', 'astoundingly', 'breathtakingly',
    'spectacular', 'phenomenal', 'incredible', 'unreal', 'surreal', 'mind-blowing',
    'greatest', 'best', 'worst', 'perfect', 'flawless', 'genius', 'brilliant',
    'amazing', 'fantastic', 'enormous', 'massive', 'gigantic', 'monumental',
    'unprecedented', 'revolutionary', 'earth-shattering', 'game-changing',
    'perfectly', 'beautifully', 'wonderfully', 'gloriously',
}

POSITIVE_WORDS = {
    'good', 'great', 'wonderful', 'excellent', 'superb', 'outstanding', 'brilliant',
    'fantastic', 'amazing', 'incredible', 'perfect', 'best', 'beautiful', 'love',
    'enjoy', 'happy', 'glad', 'pleased', 'delighted', 'celebrate', 'praise',
    'honor', 'award', 'heroic', 'noble', 'brave', 'generous', 'kind', 'helpful',
    'inspiring', 'admire', 'commend', 'applaud', 'heartwarming', 'joyful',
    'thrilled', 'excited', 'elated', 'overjoyed', 'blessed',
}

NEGATIVE_ENTITIES = {
    'trump', 'isis', 'taliban', 'congress', 'senate', 'politician', 'politicians',
    'government', 'lobbyist', 'ceo', 'corporate', 'corporation', 'bank', 'banks',
    'insurance', 'pharmaceutical', 'lawyer', 'lawyers', 'bureaucrat', 'bureaucrats',
    'republican', 'democrat', 'republicans', 'democrats',
}

ONION_PATTERNS = [
    r'\barea\s+(man|woman|teen|child|boy|girl|resident|father|mother)\b',
    r'\blocal\s+(man|woman|teen|child|boy|girl|resident|father|mother)\b',
    r'\bnation.?s?\s+(man|woman|children|citizens|workers|taxpayers|seniors)\b',
    r'\breport(s|edly)?:',
    r'\bstudies?\s+show\b',
    r'\bnew\s+report\b',
    r'\bexperts?\s+(say|claim|warn|reveal|suggest|confirm|agree)\b',
    r'\bscientists?\s+(say|claim|warn|reveal|suggest|discover|confirm)\b',
    r'\breport\s+finds?\b',
    r'\bnation.?s\b',
]

INCONGRUITY_PAIRS = [
    ({'excited', 'thrilled', 'happy', 'glad', 'pleased', 'delighted', 'overjoyed'},
     {'cancer', 'death', 'dying', 'bankrupt', 'fired', 'killed', 'arrested',
      'disaster', 'war', 'bomb', 'shooting', 'crash', 'poverty', 'recession',
      'unemployed', 'evicted', 'homeless', 'disease', 'pandemic'}),
    ({'celebrate', 'celebrates', 'honor', 'honors', 'awards', 'praises'},
     {'failure', 'loss', 'defeat', 'worst', 'mistake', 'error', 'blunder', 'flop'}),
]

print('✅ Lexicons loaded.')
print(f'   Hyperbole words   : {len(HYPERBOLE_WORDS)}')
print(f'   Positive words    : {len(POSITIVE_WORDS)}')
print(f'   Negative entities : {len(NEGATIVE_ENTITIES)}')
print(f'   Onion patterns    : {len(ONION_PATTERNS)}')

In [ ]:
# ── Rule scoring function ─────────────────────────────────────────────────────

def score_headline(text: str) -> dict:
    """
    Score a headline using 7 linguistic signals.
    Returns dict with keys: signals, score, prediction.
    Decision threshold: score >= 2 → sarcastic (label=1).
    """
    doc      = nlp(text)
    tl       = {t.text.lower() for t in doc}    # token-level lowercase
    ll       = {t.lemma_.lower() for t in doc}  # lemma-level lowercase
    text_low = text.lower()

    signals = {}

    # ── Signal 1: Hyperbole (≥2 extreme intensifiers / superlatives) ──────────
    hyp_hits = tl & HYPERBOLE_WORDS
    signals['hyperbole'] = 1 if len(hyp_hits) >= 2 else 0

    # ── Signal 2: Ironic quotation marks ──────────────────────────────────────
    signals['ironic_quotes'] = 1 if any(
        c in text for c in ['"', "'", '\u2018', '\u2019', '\u201c', '\u201d']
    ) else 0

    # ── Signal 3: Exclamation mark ────────────────────────────────────────────
    signals['exclamation'] = 1 if '!' in text else 0

    # ── Signal 4: The Onion structural pattern ────────────────────────────────
    signals['onion_pattern'] = 1 if any(
        re.search(p, text_low) for p in ONION_PATTERNS
    ) else 0

    # ── Signal 5: Sentiment incongruity (positive word + negative named entity) 
    has_pos     = bool(ll & POSITIVE_WORDS)
    ner_tokens  = {ent.text.lower() for ent in doc.ents} | tl
    has_neg_ent = bool(ner_tokens & NEGATIVE_ENTITIES)
    signals['sentiment_incongruity'] = 1 if (has_pos and has_neg_ent) else 0

    # ── Signal 6: Negation within 2-token window of positive adjective ────────
    neg_idx = {i for i, t in enumerate(doc) if t.lower_ in {'not', "n't", 'never', 'no'}}
    pos_idx = {i for i, t in enumerate(doc) if t.lemma_.lower() in POSITIVE_WORDS}
    signals['negation_positive'] = 1 if any(
        abs(ni - pi) <= 2 for ni in neg_idx for pi in pos_idx
    ) else 0

    # ── Signal 7: Explicit incongruity pair (positive emotion + catastrophe) ──
    explicit = 0
    for pos_set, neg_set in INCONGRUITY_PAIRS:
        if (ll & pos_set) and (ll & neg_set):
            explicit = 1
            break
    signals['explicit_incongruity'] = explicit

    total_score = sum(signals.values())
    prediction  = 1 if total_score >= 2 else 0

    return {'signals': signals, 'score': total_score, 'prediction': prediction}


# ── Sanity checks ─────────────────────────────────────────────────────────────
sanity_tests = [
    ("Area Man Passionate Defender Of What He Imagines Constitution To Be",      1),
    ("Scientists discover new species of fish in the Pacific Ocean",              0),
    ("Trump absolutely brilliant at solving nation's healthcare crisis",           1),
    ("New drug shows promise for treating Alzheimer's disease",                   0),
    ("Nation's Obesity Rates Skyrocket As McDonald's Introduces 4000-Cal Salad", 1),
    ("Federal Reserve raises interest rates for third time this year",            0),
]
print('Sanity checks:')
correct = 0
for headline, true_label in sanity_tests:
    res = score_headline(headline)
    ok  = '✅' if res['prediction'] == true_label else '❌'
    if res['prediction'] == true_label: correct += 1
    print(f'  {ok} [TRUE={true_label} PRED={res["prediction"]} SCORE={res["score"]}]',
          headline[:70])
print(f'\nSanity accuracy: {correct}/{len(sanity_tests)}')
print('✅ score_headline() defined.')

## Section 4 — Predict on Test Set

In [ ]:
# ── Run rule engine on every test headline ────────────────────────────────────
print(f'⏳ Scoring {len(X_test):,} test headlines with rule engine...')
t0 = time.time()

results = [
    score_headline(headline)
    for headline in tqdm(X_test, desc='Rule scoring')
]

# Extract arrays — naming mirrors 02_tfidf_sklearn.ipynb conventions
y_pred  = np.array([r['prediction'] for r in results])
y_score = np.array([r['score']      for r in results], dtype=float)

elapsed = time.time() - t0
print(f'   Done in {elapsed:.1f}s  ({elapsed/len(X_test)*1000:.2f} ms per headline)')
print(f'\nPredicted sarcastic : {y_pred.sum():,} ({y_pred.mean()*100:.1f}%)')
print(f'True sarcastic      : {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

## Section 5 — Full Evaluation (10 Rubric Metrics)

> Uses **the exact same `compute_all_metrics()` function** as `02_tfidf_sklearn.ipynb` and
> `03_bert_finetune.ipynb` — same formula, same variable names, same print format,
> same return dict structure.

In [ ]:
# ── compute_all_metrics — identical to 02_tfidf_sklearn.ipynb ─────────────────

def compute_all_metrics(y_true, y_pred, y_prob=None, model_name='Model'):
    """
    Compute all 10 evaluation metrics from the CSE3246 rubric.
    Returns a dict of metrics and prints a formatted report.
    """
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    P = TP + FN   # Total actual positives
    N = TN + FP   # Total actual negatives

    # ── All 10 required metrics ───────────────────────────────────────────────
    accuracy    = (TP + TN) / (TP + TN + FP + FN)
    precision   = TP / (FP + TP) if (FP + TP) > 0 else 0.0
    recall      = TP / (FN + TP) if (FN + TP) > 0 else 0.0   # = Sensitivity
    f1          = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    sensitivity = TP / P if P > 0 else 0.0                    # TPR
    specificity = TN / N if N > 0 else 0.0                    # TNR
    fpr         = FP / N if N > 0 else 0.0
    fnr         = FN / P if P > 0 else 0.0
    npv         = TN / (TN + FN) if (TN + FN) > 0 else 0.0
    fdr         = FP / (FP + TP) if (FP + TP) > 0 else 0.0
    mcc         = matthews_corrcoef(y_true, y_pred)

    roc_auc = roc_auc_score(y_true, y_prob) if y_prob is not None else None

    metrics = {
        'model': model_name,
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'Accuracy':    round(accuracy,    4),
        'Precision':   round(precision,   4),
        'Recall':      round(recall,      4),
        'F1':          round(f1,          4),
        'Sensitivity': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'FPR':         round(fpr,         4),
        'FNR':         round(fnr,         4),
        'NPV':         round(npv,         4),
        'FDR':         round(fdr,         4),
        'MCC':         round(mcc,         4),
        'ROC_AUC':     round(roc_auc, 4) if roc_auc else 'N/A',
    }

    print(f'\n{"="*55}')
    print(f'  📊  {model_name} — Evaluation Results')
    print(f'{"="*55}')
    print(f'  Confusion Matrix  → TP:{TP:5d} | TN:{TN:5d} | FP:{FP:5d} | FN:{FN:5d}')
    print(f'{"─"*55}')
    rubric_metrics = ['Accuracy','Precision','Recall','F1','Sensitivity',
                      'Specificity','FPR','FNR','NPV','FDR','MCC']
    for i, m in enumerate(rubric_metrics, 1):
        bar = '█' * int(metrics[m] * 20)
        print(f'  {i:2d}. {m:14s}: {metrics[m]:.4f}  {bar}')
    if roc_auc:
        print(f'  ✨  ROC-AUC      : {roc_auc:.4f}')
    print(f'{"="*55}')
    return metrics

print('✅ compute_all_metrics() defined.')

In [ ]:
# ── Evaluate rule-based predictions ──────────────────────────────────────────
# y_score (raw integer 0-7) used as soft probability proxy for ROC-AUC.
# This is appropriate because higher scores indicate stronger sarcasm signal.

metrics_rb = compute_all_metrics(
    y_test,
    y_pred,
    y_prob=y_score,       # soft score as prob proxy
    model_name='Rule-Based (spaCy, threshold=2)'
)

## Section 6 — Visualisations

In [ ]:
# ── Plot 1: Confusion Matrix heatmap — same style as eval_01_confusion_matrices.png

cm_val = confusion_matrix(y_test, y_pred)
cm_pct = cm_val / cm_val.sum(axis=1, keepdims=True) * 100
annot  = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(rv, rp)]
                   for rv, rp in zip(cm_val, cm_pct)])
class_names = ['Non-Sarcastic', 'Sarcastic']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_val, annot=annot, fmt='', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Confusion Matrix — Method 1: Rule-Based (spaCy)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label',      fontsize=11)
ax.set_xticklabels(class_names, rotation=15)
ax.set_yticklabels(class_names, rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rb_eval_01_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: figures/rb_eval_01_confusion_matrix.png')

In [ ]:
# ── Plot 2: All 10 Metrics — Bar Chart ───────────────────────────────────────
# Green = higher is better; Red = lower is better (FPR, FNR, FDR)

METRIC_COLS = ['Accuracy','Precision','Recall','F1','Sensitivity',
               'Specificity','FPR','FNR','NPV','FDR','MCC']
vals   = [metrics_rb[m] for m in METRIC_COLS]
colors = ['#2ecc71' if m not in ('FPR','FNR','FDR') else '#e74c3c' for m in METRIC_COLS]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(METRIC_COLS, vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold', rotation=90)
ax.set_xticks(range(len(METRIC_COLS)))
ax.set_xticklabels(METRIC_COLS, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Metric Score', fontsize=12)
ax.set_title('All 10 Evaluation Metrics — Method 1: Rule-Based (spaCy)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.18)
ax.axhline(0.5, color='gray', ls='--', alpha=0.4, lw=1)
ax.grid(axis='y', alpha=0.3)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', alpha=0.85, label='Higher = better'),
                   Patch(facecolor='#e74c3c', alpha=0.85, label='Lower = better (error rates)')]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rb_eval_02_metrics_bar.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: figures/rb_eval_02_metrics_bar.png')

In [ ]:
# ── Plot 3: Signal Activation Rate by Prediction Outcome ─────────────────────

signal_names = list(results[0]['signals'].keys())

tp_mask = (y_test == 1) & (y_pred == 1)   # Correctly detected sarcasm
fp_mask = (y_test == 0) & (y_pred == 1)   # Over-triggered on real news
fn_mask = (y_test == 1) & (y_pred == 0)   # Missed sarcasm

def signal_rate(mask):
    subset = [results[i]['signals'] for i in range(len(results)) if mask[i]]
    return [np.mean([s[sn] for s in subset]) for sn in signal_names] if subset else [0.0]*len(signal_names)

tp_rates = signal_rate(tp_mask)
fp_rates = signal_rate(fp_mask)
fn_rates = signal_rate(fn_mask)

x, w = np.arange(len(signal_names)), 0.26
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, tp_rates, w, label='True Positive (caught sarcasm)',    color='#2ecc71', alpha=0.85)
ax.bar(x,     fp_rates, w, label='False Positive (over-triggered)',  color='#e74c3c', alpha=0.85)
ax.bar(x + w, fn_rates, w, label='False Negative (missed sarcasm)',  color='#f39c12', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(signal_names, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Signal Activation Rate', fontsize=12)
ax.set_title('Rule Signal Activation Rate by Prediction Outcome — Method 1',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rb_eval_03_signal_rates.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: figures/rb_eval_03_signal_rates.png')
print('\n💡 onion_pattern fires most selectively on true positives.')
print('   sentiment_incongruity and hyperbole also fire on FP → real news uses intensifiers too.')

In [ ]:
# ── Plot 4: Score Distribution by True Label ──────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.arange(-0.5, y_score.max() + 1.5, 1)
ax.hist(y_score[y_test == 0], bins=bins, alpha=0.65, density=True,
        label='Non-Sarcastic', color='#4A90D9')
ax.hist(y_score[y_test == 1], bins=bins, alpha=0.65, density=True,
        label='Sarcastic',     color='#E74C3C')
ax.axvline(1.5, color='black', ls='--', lw=2, label='Decision threshold (score ≥ 2)')
ax.set_xlabel('Rule Score (0–7 signals fired)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Rule Score Distribution by True Label — Method 1',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rb_eval_04_score_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: figures/rb_eval_04_score_distribution.png')
print('\n💡 Most non-sarcastic headlines score 0–1; sarcastic pile up at 2+.')
print('   Overlap at score=1 is irreducible noise for any rule-based threshold.')

## Section 7 — Error Analysis

> Same structure as Section 8 in `02_tfidf_sklearn.ipynb`.

In [ ]:
# ── Error Analysis — understand failure modes ─────────────────────────────────

test_df = pd.DataFrame({
    'headline':   X_test,
    'true_label': y_test,
    'rb_pred':    y_pred,
    'rb_score':   y_score.astype(int),
    'lr_prob':    y_score / max(y_score.max(), 1),  # normalised — same field name as 02 notebook
})

fp_df = test_df[(test_df['true_label'] == 0) & (test_df['rb_pred'] == 1)].nlargest(8, 'rb_score')
fn_df = test_df[(test_df['true_label'] == 1) & (test_df['rb_pred'] == 0)].nsmallest(8, 'lr_prob')

print('❌ FALSE POSITIVES — predicted SARCASTIC, actually NON-SARCASTIC')
print('   (Real news that sounds hyperbolic or contains Onion-like patterns):')
for _, row in fp_df.iterrows():
    print(f'   [score={row["rb_score"]}] {row["headline"][:100]}')

print('\n❌ FALSE NEGATIVES — predicted NON-SARCASTIC, actually SARCASTIC')
print('   (Subtle Onion sarcasm with no explicit lexical cue — needs BERT context):')
for _, row in fn_df.iterrows():
    print(f'   [score={row["rb_score"]}] {row["headline"][:100]}')

print('\n💡 Key insight: Rule-based misses implicit/culturally-grounded sarcasm.')
print('   These cases require statistical pattern learning (M2) or deep context (M3: BERT).')

## Section 8 — Export JSON

> Output matches the structure expected by `03_bert_finetune.ipynb` Section 10.
> The `metrics` key uses the same field names as `method2_tfidf_results.json`.

In [ ]:
# ── Compile and save ──────────────────────────────────────────────────────────

method1_results = {
    'method':       'Method 1 — Rule-Based (spaCy)',
    'dataset':      'raquiba/Sarcasm_News_Headline',
    'dataset_size': int(len(df)),
    'train_size':   int(len(X_train)),
    'test_size':    int(len(X_test)),
    'approach':     'Linguistic rule engine — no training required',
    'spacy_model':  'en_core_web_sm',
    'decision_threshold': 2,
    'signals_used': [
        'hyperbole',
        'ironic_quotes',
        'exclamation',
        'onion_pattern',
        'sentiment_incongruity',
        'negation_positive',
        'explicit_incongruity',
    ],
    'metrics': metrics_rb,
    'figures': [
        'figures/rb_eval_01_confusion_matrix.png',
        'figures/rb_eval_02_metrics_bar.png',
        'figures/rb_eval_03_signal_rates.png',
        'figures/rb_eval_04_score_distribution.png',
    ],
}

json_path = M1_DIR / 'method1_rulebased_results.json'
with open(json_path, 'w') as f:
    json.dump(method1_results, f, indent=2)

print(f'✅ Results saved → {json_path}')
print(f'\n📋 Quick summary card:')
print(f'   Accuracy : {metrics_rb["Accuracy"]:.4f}')
print(f'   Precision: {metrics_rb["Precision"]:.4f}')
print(f'   Recall   : {metrics_rb["Recall"]:.4f}')
print(f'   F1       : {metrics_rb["F1"]:.4f}')
print(f'   MCC      : {metrics_rb["MCC"]:.4f}')
print(f'   ROC-AUC  : {metrics_rb["ROC_AUC"]}')

In [ ]:
# ── Final printable metrics table — same format as 02_tfidf_sklearn.ipynb ────

METRIC_COLS_DISPLAY = ['Accuracy','Precision','Recall','F1','Sensitivity',
                       'Specificity','FPR','FNR','NPV','FDR','MCC','ROC_AUC']

summary_table = pd.DataFrame([{
    'Method': 'Method 1 — Rule-Based (spaCy)',
    **{m: metrics_rb[m] for m in METRIC_COLS_DISPLAY}
}]).set_index('Method')

print('\n' + '='*80)
print('  FINAL METRICS TABLE — Method 1 (Rule-Based spaCy)')
print('='*80)
print(summary_table.to_string())
print('='*80)
print('\n✅ Notebook complete! All outputs saved to ./outputs/')
print('   → Next: Run 02_tfidf_sklearn.ipynb (Method 2)')
print('   → Then: Run 03_bert_finetune.ipynb (Method 3)')